In [4]:
import tkinter as tk
from tkinter import messagebox
import random

class NameDrawerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("🎉 Presentation Name Drawer 🎉")
        self.root.geometry("1000x700")
        self.root.configure(bg="#FFF7E6")

        self.original_names = []
        self.remaining_names = []
        self.is_animating = False
        self.confetti_items = []

        self.bg_color = "#FFF7E6"
        self.card_color = "#FFFFFF"
        self.title_color = "#7C3AED"
        self.button_colors = {
            "load": "#60A5FA",
            "draw": "#34D399",
            "reset": "#FBBF24",
        }

        self.build_ui()

    def build_ui(self):
        title = tk.Label(
            self.root,
            text="🎉 Presentation Name Drawer 🎉",
            font=("Arial", 30, "bold"),
            bg=self.bg_color,
            fg=self.title_color
        )
        title.pack(pady=20)

        instruction = tk.Label(
            self.root,
            text="Enter names separated by commas or new lines:",
            font=("Arial", 16, "bold"),
            bg=self.bg_color,
            fg="#374151"
        )
        instruction.pack(pady=5)

        self.entry = tk.Text(
            self.root,
            height=5,
            width=65,
            font=("Arial", 14),
            relief="flat",
            bd=3,
            bg="#FFFDF8",
            fg="#111827",
            highlightthickness=2,
            highlightbackground="#F59E0B",
            highlightcolor="#F59E0B"
        )
        self.entry.pack(pady=10)

        button_frame = tk.Frame(self.root, bg=self.bg_color)
        button_frame.pack(pady=12)

        self.load_button = tk.Button(
            button_frame,
            text="Load Names",
            font=("Arial", 14, "bold"),
            width=14,
            command=self.load_names,
            bg=self.button_colors["load"],
            fg="white",
            activebackground="#3B82F6",
            relief="flat",
            padx=10,
            pady=8
        )
        self.load_button.grid(row=0, column=0, padx=10)

        self.draw_button = tk.Button(
            button_frame,
            text="Draw Name",
            font=("Arial", 14, "bold"),
            width=14,
            command=self.start_draw,
            bg=self.button_colors["draw"],
            fg="white",
            activebackground="#10B981",
            relief="flat",
            padx=10,
            pady=8
        )
        self.draw_button.grid(row=0, column=1, padx=10)

        self.reset_button = tk.Button(
            button_frame,
            text="Reset",
            font=("Arial", 14, "bold"),
            width=14,
            command=self.reset_names,
            bg=self.button_colors["reset"],
            fg="white",
            activebackground="#F59E0B",
            relief="flat",
            padx=10,
            pady=8
        )
        self.reset_button.grid(row=0, column=2, padx=10)

        self.result_frame = tk.Frame(
            self.root,
            bg=self.card_color,
            highlightthickness=4,
            highlightbackground="#EC4899"
        )
        self.result_frame.pack(pady=25)

        self.result_label = tk.Label(
            self.result_frame,
            text="Ready to draw!",
            font=("Arial", 38, "bold"),
            bg=self.card_color,
            fg="#111827",
            width=24,
            height=3
        )
        self.result_label.pack(padx=20, pady=20)

        self.status_label = tk.Label(
            self.root,
            text="No names loaded.",
            font=("Arial", 16, "bold"),
            bg=self.bg_color,
            fg="#4B5563"
        )
        self.status_label.pack(pady=8)

        self.names_left_label = tk.Label(
            self.root,
            text="Remaining names: 0",
            font=("Arial", 20, "bold"),
            bg=self.bg_color,
            fg="#2563EB"
        )
        self.names_left_label.pack(pady=5)

        self.canvas = tk.Canvas(
            self.root,
            width=950,
            height=220,
            bg=self.bg_color,
            highlightthickness=0
        )
        self.canvas.pack(pady=10)

    def parse_names(self):
        raw_text = self.entry.get("1.0", tk.END)
        names = [name.strip() for name in raw_text.replace("\n", ",").split(",")]
        return [name for name in names if name]

    def load_names(self):
        names = self.parse_names()
        if not names:
            messagebox.showwarning("No Names", "Please enter at least one name.")
            return

        self.original_names = names[:]
        self.remaining_names = names[:]
        self.result_label.config(text="Names loaded!", fg="#7C3AED")
        self.status_label.config(text="Click 'Draw Name' to begin.")
        self.update_remaining_count()
        self.clear_confetti()

    def start_draw(self):
        if self.is_animating:
            return

        if not self.remaining_names:
            messagebox.showinfo("Done", "No names left to draw. Click Reset or load new names.")
            return

        self.is_animating = True
        self.draw_button.config(state="disabled")
        self.load_button.config(state="disabled")
        self.clear_confetti()
        self.animate_shuffle(0, 24)

    def animate_shuffle(self, count, max_count):
        if count < max_count:
            fake_name = random.choice(self.remaining_names)
            colors = ["#EF4444", "#F59E0B", "#10B981", "#3B82F6", "#8B5CF6", "#EC4899"]
            self.result_label.config(text=fake_name, fg=random.choice(colors))
            delay = 60 + count * 8
            self.root.after(delay, lambda: self.animate_shuffle(count + 1, max_count))
        else:
            winner = random.choice(self.remaining_names)
            self.remaining_names.remove(winner)

            self.result_label.config(text=winner, fg="#DC2626")
            self.status_label.config(text=f"🎊 Drawn: {winner} 🎊")
            self.update_remaining_count()

            self.is_animating = False
            self.draw_button.config(state="normal")
            self.load_button.config(state="normal")

            self.launch_confetti()

            if not self.remaining_names:
                self.status_label.config(text=f"🎉 Drawn: {winner} | All names have been used.")

    def reset_names(self):
        if not self.original_names:
            messagebox.showwarning("Nothing to Reset", "Load names first.")
            return

        self.remaining_names = self.original_names[:]
        self.result_label.config(text="Reset complete!", fg="#7C3AED")
        self.status_label.config(text="All names are back in the draw.")
        self.update_remaining_count()
        self.clear_confetti()

    def update_remaining_count(self):
        self.names_left_label.config(text=f"Remaining names: {len(self.remaining_names)}")

    def clear_confetti(self):
        for item in self.confetti_items:
            self.canvas.delete(item)
        self.confetti_items = []

    def launch_confetti(self):
        self.clear_confetti()

        confetti_colors = [
            "#EF4444", "#F59E0B", "#FDE047",
            "#22C55E", "#06B6D4", "#3B82F6",
            "#8B5CF6", "#EC4899"
        ]

        pieces = []
        for _ in range(120):
            x = random.randint(20, 930)
            y = random.randint(-180, -10)
            size = random.randint(6, 14)
            color = random.choice(confetti_colors)
            shape_type = random.choice(["oval", "rect"])

            if shape_type == "oval":
                item = self.canvas.create_oval(x, y, x + size, y + size, fill=color, outline="")
            else:
                item = self.canvas.create_rectangle(x, y, x + size, y + size, fill=color, outline="")

            dx = random.randint(-3, 3)
            dy = random.randint(4, 9)
            pieces.append((item, dx, dy))

        self.confetti_items = [p[0] for p in pieces]
        self.animate_confetti(pieces, 0)

    def animate_confetti(self, pieces, step):
        if step > 45:
            return

        new_pieces = []
        for item, dx, dy in pieces:
            self.canvas.move(item, dx, dy)
            coords = self.canvas.coords(item)

            if coords and coords[1] < 240:
                new_pieces.append((item, dx, dy + 0.2))

        self.root.after(40, lambda: self.animate_confetti(new_pieces, step + 1))


if __name__ == "__main__":
    root = tk.Tk()
    app = NameDrawerApp(root)
    root.mainloop()